# TRACK-FA Raw Combination and Drop3POMs Missingness Audit

This notebook is table-only. It checks missing entities, missing columns, and missing visits at two stages:

1. Direct raw REDCap + raw MasterFile entry join by `participant_id` and `visit`.
2. Final modelling dataset `trackfa_pairs_drop3poms.csv`.

`trackfa_pairs_drop3poms.csv` is treated as the correct paired modelling CSV. In that file, `feature_baseline` is the V1 or V2 value, `feature_followup` is the V2 or V3 value, and `delta_feature` is either V2-V1 or V3-V2, depending on `patient_id` suffix `V1V2` or `V2V3`.


In [1]:
from __future__ import annotations

import ast
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    for path in (start.resolve(), *start.resolve().parents):
        if (path / "src").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not find the project root containing src/ and data/.")

REPO_ROOT = find_project_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.trackfa import raw_direct_combination_audit

PAIRS_PATH = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
if not PAIRS_PATH.exists():
    raise FileNotFoundError(f"Required modelling dataset not found: {PAIRS_PATH}")

pairs_df = pd.read_csv(PAIRS_PATH)
print(f"Loaded required modelling CSV: {PAIRS_PATH}")
print(f"Shape: {pairs_df.shape[0]} rows x {pairs_df.shape[1]} columns")


Loaded required modelling CSV: /Users/robertwang/Documents/New_project/biomarkers/data/processed/trackfa_pairs_drop3poms.csv
Shape: 207 rows x 455 columns


## 1. Direct Raw Entry Join Missingness


In [2]:
raw_audit = raw_direct_combination_audit()
raw_combined = raw_audit["combined"]

print("Raw source sizes and direct outer-join size")
display(raw_audit["source_summary"])

print("Missing entities/rows by source presence")
display(raw_audit["presence_summary"])

print("Missing visits by source presence")
display(raw_audit["visit_presence"])

print("Missing columns by raw source block")
display(raw_audit["block_summary"])

raw_missing_columns = raw_audit["missingness"].query("missing_pct > 0").copy()
print(f"Columns with any missingness after direct raw join: {len(raw_missing_columns)}")
display(raw_missing_columns)

raw_missing_by_visit = raw_audit["missingness_by_visit"].query("missing_pct > 0").copy()
print(f"Column x visit missingness rows after direct raw join: {len(raw_missing_by_visit)}")
display(raw_missing_by_visit.sort_values(["missing_pct", "visit", "column"], ascending=[False, True, True], kind="mergesort"))


Raw source sizes and direct outer-join size


,source,raw_rows,raw_columns,visit_rows_used_for_direct_join,unique_participants
0,REDCap raw export,1076,483,807,269
1,Imaging MasterFile all sheets,793,157,747,269
2,Direct outer join,807,639,807,269


Missing entities/rows by source presence


,source_presence,n_rows
0,both,747
1,left_only,60
2,right_only,0


Missing visits by source presence


...

,source_block,n_columns,mean_missing_pct,max_missing_pct
0,clinical_raw,481,0.805488,1.00000
1,imaging_raw,155,0.206516,0.39653


Columns with any missingness after direct raw join: 635


,column,missing_pct,source_block
0,clinical__age,1.000000,clinical_raw
1,clinical__age_group,1.000000,clinical_raw
2,clinical__at_school,1.000000,clinical_raw
3,clinical__disease_duration,1.000000,clinical_raw
4,clinical__education_highest_level,1.000000,clinical_raw
...,...,...,...
630,imaging__Cereb_vol,0.105328,imaging_raw
631,imaging__csa_c1c2,0.085502,imaging_raw
632,imaging__sCSA_C12_UMN,0.080545,imaging_raw
633,imaging__Protocol,0.074349,imaging_raw


Column x visit missingness rows after direct raw join: 1841


,visit,column,missing_pct,source_block
753,1,clinical__adl_assessor_v2,1.000000,clinical_raw
795,1,clinical__adl_assessor_v3,1.000000,clinical_raw
780,1,clinical__adl_bladder_v2,1.000000,clinical_raw
822,1,clinical__adl_bladder_v3,1.000000,clinical_raw
750,1,clinical__adl_date_v2,1.000000,clinical_raw
...,...,...,...,...
1449,1,imaging__csa_c1c2,0.011152,imaging_raw
1368,1,clinical__upenn_sample_time,0.003717,clinical_raw
1446,1,imaging__Protocol,0.003717,imaging_raw
1443,1,imaging__ScanDate,0.003717,imaging_raw


## 2. Final `trackfa_pairs_drop3poms.csv` Entity, Column, and Visit Audit


In [3]:
pair_meta = pairs_df["patient_id"].astype(str).str.extract(r"(?P<subject>.+)_(?P<pair_type>V1V2|V2V3)$")
if pair_meta.isna().any().any():
    bad = pairs_df.loc[pair_meta.isna().any(axis=1), "patient_id"].head(10).tolist()
    raise ValueError(f"Could not parse patient_id suffix for rows such as: {bad}")

pair_audit = pd.DataFrame({
    "n_pair_rows": [len(pairs_df)],
    "n_unique_subjects": [pair_meta["subject"].nunique()],
    "n_v1v2_rows": [int((pair_meta["pair_type"] == "V1V2").sum())],
    "n_v2v3_rows": [int((pair_meta["pair_type"] == "V2V3").sum())],
    "n_subjects_with_both_intervals": [int(pair_meta.groupby("subject")["pair_type"].nunique().eq(2).sum())],
    "n_subjects_with_one_interval": [int(pair_meta.groupby("subject")["pair_type"].nunique().eq(1).sum())],
})
print("Final pair/entity counts")
display(pair_audit)

subject_interval_sets = pair_meta.groupby("subject")["pair_type"].apply(lambda s: set(s)).reset_index(name="observed_intervals")
subject_interval_sets["missing_v1v2"] = ~subject_interval_sets["observed_intervals"].apply(lambda s: "V1V2" in s)
subject_interval_sets["missing_v2v3"] = ~subject_interval_sets["observed_intervals"].apply(lambda s: "V2V3" in s)
missing_interval_subjects = subject_interval_sets[subject_interval_sets[["missing_v1v2", "missing_v2v3"]].any(axis=1)].copy()
print(f"Subjects missing one adjacent interval in drop3poms: {len(missing_interval_subjects)}")
display(missing_interval_subjects)

missing_columns = (
    pairs_df.isna().mean().rename("missing_pct").reset_index().rename(columns={"index": "column"})
)
missing_columns["n_missing"] = pairs_df.isna().sum().values
missing_columns = missing_columns[missing_columns["n_missing"] > 0].sort_values(
    ["missing_pct", "column"], ascending=[False, True], kind="mergesort"
)
print(f"Columns with missing values in drop3poms: {len(missing_columns)}")
display(missing_columns)

missing_by_pair_type = (
    pairs_df.assign(pair_type=pair_meta["pair_type"])
    .groupby("pair_type", dropna=False)
    .apply(lambda g: g.drop(columns=["pair_type"]).isna().mean())
    .reset_index()
    .melt(id_vars="pair_type", var_name="column", value_name="missing_pct")
)
missing_by_pair_type = missing_by_pair_type[missing_by_pair_type["missing_pct"] > 0].sort_values(
    ["missing_pct", "pair_type", "column"], ascending=[False, True, True], kind="mergesort"
)
print(f"Column x pair-type missingness rows in drop3poms: {len(missing_by_pair_type)}")
display(missing_by_pair_type)


Final pair/entity counts


,n_pair_rows,n_unique_subjects,n_v1v2_rows,n_v2v3_rows,n_subjects_with_both_intervals,n_subjects_with_one_interval
0,207,117,108,99,90,27


Subjects missing one adjacent interval in drop3poms: 27


,subject,observed_intervals,missing_v1v2,missing_v2v3
1,AAN003,{V1V2},False,True
17,AAN037,{V1V2},False,True
19,AAN042,{V1V2},False,True
20,AAN045,{V2V3},True,False
34,CHP014,{V1V2},False,True
37,CHP020,{V2V3},True,False
41,CHP030,{V1V2},False,True
43,CHP040,{V1V2},False,True
46,CHP047,{V1V2},False,True
49,CHP051,{V2V3},True,False


Columns with missing values in drop3poms: 0


,column,missing_pct,n_missing


Column x pair-type missingness rows in drop3poms: 0


/var/folders/2y/d2x11n9s4sbc3j2svdb5qdb00000gn/T/ipykernel_57709/811707489.py:37: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.drop(columns=["pair_type"]).isna().mean())


,pair_type,column,missing_pct


## 3. Excluded Columns and Feature Bases


In [4]:
wide_path = REPO_ROOT / "data" / "processed" / "trackfa_merged_wide.csv"
wide_cols = list(pd.read_csv(wide_path, nrows=1).columns)
pair_cols = list(pairs_df.columns)
wide_visit_bases = {re.sub(r"_v[123]$", "", c) for c in wide_cols if re.search(r"_v[123]$", c)}
pair_bases = set()
for c in pair_cols:
    if c.endswith("_baseline"):
        pair_bases.add(c[: -len("_baseline")])
    elif c.endswith("_followup"):
        pair_bases.add(c[: -len("_followup")])
    elif c.startswith("delta_"):
        pair_bases.add(c[len("delta_"):])

excluded_visit_bases = sorted(wide_visit_bases - pair_bases)
intentional_drop3poms = ["tNAA_myo_Ins", "DN_suscept", "DN_vol"]
excluded_table = pd.DataFrame({
    "feature_or_column": excluded_visit_bases,
    "reason": [
        "intentional drop3poms exclusion" if f in intentional_drop3poms else "not part of paired modelling feature set or represented under another base name"
        for f in excluded_visit_bases
    ],
})
print("Visit-level feature bases in trackfa_merged_wide.csv not present as pair bases in drop3poms")
display(excluded_table)

removed_pair_columns = []
for base in intentional_drop3poms:
    removed_pair_columns.extend([f"{base}_baseline", f"{base}_followup", f"delta_{base}"])
print("Pair columns intentionally removed by drop3poms")
display(pd.DataFrame({"removed_pair_column": removed_pair_columns}))


Visit-level feature bases in trackfa_merged_wide.csv not present as pair bases in drop3poms


,feature_or_column,reason
0,AD_SCP_braindti,not part of paired modelling feature set or re...
1,DN_suscept,intentional drop3poms exclusion
2,DN_vol,intentional drop3poms exclusion
3,FA_SCP_braindti,not part of paired modelling feature set or re...
4,MD_SCP_braindti,not part of paired modelling feature set or re...
5,RD_SCP_braindti,not part of paired modelling feature set or re...
6,bmi,not part of paired modelling feature set or re...
7,functional_staging_score,not part of paired modelling feature set or re...
8,hpt_dom_av,not part of paired modelling feature set or re...
9,hpt_ndom_av,not part of paired modelling feature set or re...


Pair columns intentionally removed by drop3poms


,removed_pair_column
0,tNAA_myo_Ins_baseline
1,tNAA_myo_Ins_followup
2,delta_tNAA_myo_Ins
3,DN_suscept_baseline
4,DN_suscept_followup
5,delta_DN_suscept
6,DN_vol_baseline
7,DN_vol_followup
8,delta_DN_vol


## 4. Training Notebook Dataset Check


In [5]:
training_notebooks = [
    "feature_selection_pipeline.ipynb",
    "entropy_methods.ipynb",
    "comparator_table.ipynb",
    "srm_composite.ipynb",
    "progression_dl.ipynb",
    "interaction_term.ipynb",
    "model_performance.ipynb",
]
usage_rows = []
for name in training_notebooks:
    path = REPO_ROOT / "notebooks" / name
    text = path.read_text()
    uses_drop3poms = "trackfa_pairs_drop3poms.csv" in text
    fallback_to_strict = "trackfa_pairs.csv" in text and "trackfa_pairs_drop3poms.csv" in text and "if not" in text
    uses_strict_directly = "trackfa_pairs.csv" in text and "trackfa_pairs_drop3poms.csv" not in text
    usage_rows.append({
        "notebook": name,
        "uses_trackfa_pairs_drop3poms": uses_drop3poms,
        "has_fallback_to_trackfa_pairs_csv": fallback_to_strict,
        "uses_trackfa_pairs_csv_directly": uses_strict_directly,
        "status": "OK" if uses_drop3poms and not fallback_to_strict and not uses_strict_directly else "CHECK",
    })
usage_table = pd.DataFrame(usage_rows)
display(usage_table)
if (usage_table["status"] != "OK").any():
    raise AssertionError("At least one training notebook is not pinned cleanly to trackfa_pairs_drop3poms.csv")


,notebook,uses_trackfa_pairs_drop3poms,has_fallback_to_trackfa_pairs_csv,uses_trackfa_pairs_csv_directly,status
0,feature_selection_pipeline.ipynb,True,False,False,OK
1,entropy_methods.ipynb,True,False,False,OK
2,comparator_table.ipynb,True,False,False,OK
3,srm_composite.ipynb,True,False,False,OK
4,progression_dl.ipynb,True,False,False,OK
5,interaction_term.ipynb,True,False,False,OK
6,model_performance.ipynb,True,False,False,OK
